In [1]:
# ============================================================
# STEP 1: MOUNT GOOGLE DRIVE
# ============================================================
# This notebook reads the raw NHS articles stored in Google
# Drive and saves the cleaned versions back to Google Drive.
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# STEP 2: IMPORT REQUIRED LIBRARIES
# ============================================================
# pathlib  -> File and folder management
# re       -> Regular expressions for text cleaning
# tqdm     -> Progress bar
# ============================================================

from pathlib import Path
from tqdm import tqdm
import re

In [3]:
# ============================================================
# STEP 3: DEFINE PROJECT PATHS
# ============================================================
# RAW_DIR      -> Contains scraped NHS articles
# CLEANED_DIR  -> Stores cleaned articles
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Healthcare_RAG")

RAW_DIR = PROJECT_DIR / "data" / "raw"
CLEANED_DIR = PROJECT_DIR / "data" / "cleaned"

# Create the cleaned directory if it does not exist
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw Folder      :", RAW_DIR)
print("Cleaned Folder  :", CLEANED_DIR)

Raw Folder      : /content/drive/MyDrive/Healthcare_RAG/data/raw
Cleaned Folder  : /content/drive/MyDrive/Healthcare_RAG/data/cleaned


In [4]:
# ============================================================
# STEP 4: LOAD ALL RAW TEXT FILES
# ============================================================
# This cell:
#   1. Finds every .txt file inside data/raw
#   2. Stores their paths in a list
#   3. Displays a summary
# ============================================================

# Find all text files
raw_files = sorted(RAW_DIR.glob("*.txt"))

print("=" * 60)
print("RAW DATASET SUMMARY")
print("=" * 60)

print(f"Total Articles Found : {len(raw_files)}")

print("\nFirst 10 Files:")

for file in raw_files[:10]:
    print(file.name)

print("=" * 60)

RAW DATASET SUMMARY
Total Articles Found : 698

First 10 Files:
aciclovir_zovirax.txt
acrivastine-brand_name_benadryl_allergy_relief.txt
acute_lymphoblastic_leukaemia.txt
adalimumab.txt
alendronic_acid-brand_names_fosamax_binosto.txt
allopurinol-zyloric_uricto.txt
amlodipine.txt
amoxicillin.txt
anal_pain.txt
anastrozole-brand_name_arimidex.txt


In [5]:
# ============================================================
# STEP 5: DEFINE THE TEXT CLEANING FUNCTION
# ============================================================
# This function cleans the raw NHS article by:
#
# 1. Removing leading and trailing whitespace.
# 2. Removing empty lines.
# 3. Replacing multiple spaces with a single space.
# 4. Removing duplicate lines.
# 5. Preserving headings (#, ##, ###).
# 6. Preserving bullet points (•).
#
# Input:
#     Raw article text
#
# Output:
#     Cleaned article text
# ============================================================

def clean_text(text):
    """
    Clean a raw NHS article.

    Parameters
    ----------
    text : str
        Raw article text.

    Returns
    -------
    str
        Cleaned article text.
    """

    # Split the article into individual lines
    lines = text.splitlines()

    # Store cleaned lines
    cleaned_lines = []

    # Used to remove duplicate lines
    seen = set()

    # Process each line
    for line in lines:

        # Remove leading and trailing whitespace
        line = line.strip()

        # Skip empty lines
        if not line:
            continue

        # Replace multiple spaces with a single space
        line = re.sub(r"\s+", " ", line)

        # Skip duplicate lines
        if line in seen:
            continue

        # Store unique line
        seen.add(line)

        # Add cleaned line
        cleaned_lines.append(line)

    # Join all cleaned lines into one article
    cleaned_text = "\n\n".join(cleaned_lines)

    return cleaned_text

In [6]:
# ============================================================
# STEP 6: CLEAN AND SAVE ALL NHS ARTICLES
# ============================================================
# This cell:
#   1. Reads every raw article.
#   2. Cleans the article text.
#   3. Saves the cleaned article.
#   4. Tracks successful and failed files.
#   5. Displays a summary.
# ============================================================

# Statistics
success_count = 0
failed_count = 0
failed_files = []

print("=" * 70)
print("STARTING DATA CLEANING")
print("=" * 70)

# Process every text file
for file_path in tqdm(raw_files):

    try:

        # ----------------------------------------------------
        # Read raw article
        # ----------------------------------------------------
        with open(file_path, "r", encoding="utf-8") as file:
            raw_text = file.read()

        # ----------------------------------------------------
        # Clean article
        # ----------------------------------------------------
        cleaned_text = clean_text(raw_text)

        # ----------------------------------------------------
        # Save cleaned article
        # ----------------------------------------------------
        output_path = CLEANED_DIR / file_path.name

        with open(output_path, "w", encoding="utf-8") as file:
            file.write(cleaned_text)

        success_count += 1

    except Exception as error:

        failed_count += 1
        failed_files.append((file_path.name, str(error)))

# ============================================================
# CLEANING SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("DATA CLEANING COMPLETED")
print("=" * 70)

print(f"Articles Processed Successfully : {success_count}")
print(f"Failed Articles                 : {failed_count}")

if failed_files:

    print("\nFailed Files:")

    for filename, error in failed_files:
        print(f"- {filename}")
        print(f"  Error: {error}")

print("=" * 70)

STARTING DATA CLEANING


100%|██████████| 698/698 [00:23<00:00, 29.65it/s]



DATA CLEANING COMPLETED
Articles Processed Successfully : 698
Failed Articles                 : 0


In [7]:
# ============================================================
# STEP 7: VALIDATE THE CLEANING PROCESS
# ============================================================
# This cell:
#   1. Opens one raw article.
#   2. Opens its cleaned version.
#   3. Displays a preview of both.
#   4. Shows the reduction in file size.
#
# This helps verify that the cleaning process worked
# correctly before moving to the chunking stage.
# ============================================================

# Select the first article
sample_file = raw_files[0]

raw_file = RAW_DIR / sample_file.name
clean_file = CLEANED_DIR / sample_file.name

# Read raw article
with open(raw_file, "r", encoding="utf-8") as file:
    raw_text = file.read()

# Read cleaned article
with open(clean_file, "r", encoding="utf-8") as file:
    cleaned_text = file.read()

# Display comparison
print("=" * 70)
print("RAW ARTICLE PREVIEW")
print("=" * 70)
print(raw_text[:1200])

print("\n\n")

print("=" * 70)
print("CLEANED ARTICLE PREVIEW")
print("=" * 70)
print(cleaned_text[:1200])

print("\n\n")

print("=" * 70)
print("COMPARISON")
print("=" * 70)
print(f"Raw Characters     : {len(raw_text):,}")
print(f"Clean Characters   : {len(cleaned_text):,}")
print(f"Characters Removed : {len(raw_text) - len(cleaned_text):,}")
print("=" * 70)

RAW ARTICLE PREVIEW
# Aciclovir (Zovirax)

Aciclovir is an antiviral medicine used to treat certain infections. Most types are only available with a prescription, but you can buy some types from shops and pharmacies.


## What aciclovir is for

Aciclovir is used to treat infections such as:

• cold sores

• shingles

• chickenpox

• genital herpes

• herpes simplex eye infections

It's also sometimes used to prevent these infections in people with a weakened immune system.


## How to use aciclovir

There are several different types of aciclovir, including:

• tablets

• cream

• eye ointment

How you use your medicine and how much to use depends on which type it is and how much aciclovir it contains.

Always check the packet or leaflet that comes with your medicine. Ask a pharmacist or doctor for advice if you're not sure how to use it or have any problems using it.

If a doctor prescribes aciclovir for you, follow the advice they give you about how to use it and how much to use.


##